# CLIP4CAD-GFA Retrieval Evaluation

This notebook evaluates the trained CLIP4CAD-GFA model on cross-modal retrieval tasks.

**Setup:**
- Query set: Validation split
- Gallery set: Training split

**Retrieval Tasks:**
1. Text → B-Rep: Text query, B-Rep gallery
2. Text → Point Cloud: Text query, PC gallery
3. Point Cloud → B-Rep: PC query, B-Rep gallery

**Metrics:**
- mAP@k (Mean Average Precision at k)
- Recall@k (R@k)
- k ∈ {1, 5, 10, 50}

In [ ]:
# Cell 1: Imports and Setup
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import h5py
import json
from omegaconf import OmegaConf

from clip4cad.models.clip4cad_gfa import CLIP4CAD_GFA
from clip4cad.data.gfa_dataset import GFAMappedDataset, gfa_collate_fn

# Configuration
CONFIG = {
    # Paths - UPDATE THESE
    "checkpoint_path": "../outputs/gfa/checkpoint_best.pt",
    "config_path": "../configs/model/clip4cad_gfa.yaml",
    "data_root": "d:/Defect_Det/MMCAD/data",
    "mapping_dir": "d:/Defect_Det/MMCAD/data/aligned",
    "pc_file": None,  # Set if using custom PC file
    
    # Text embeddings on SSD for fast loading
    "text_splits_dir": "C:/Users/User/Desktop/text_splits",
    
    # Evaluation settings
    "batch_size": 64,
    "k_values": [1, 5, 10, 50],
    
    # Memory settings - set True for fast loading (needs ~35GB for val, ~200GB for train)
    "load_val_to_memory": True,   # Val is small (~35GB), recommended
    "load_train_to_memory": False, # Train is large (~200GB), set True if you have RAM
    
    # Output
    "output_dir": "outputs/evaluation",
}

# Set up device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Verify SSD text splits exist
text_splits_dir = Path(CONFIG["text_splits_dir"])
if text_splits_dir.exists():
    print(f"Text splits directory found: {text_splits_dir}")
    for f in text_splits_dir.glob("*.h5"):
        print(f"  - {f.name} ({f.stat().st_size / 1e9:.1f} GB)")
else:
    print(f"WARNING: Text splits directory not found: {text_splits_dir}")

In [ ]:
# Cell 2: Load Model from Checkpoint

def load_model(checkpoint_path: str, config_path: str = None):
    """Load CLIP4CAD-GFA model from checkpoint."""
    print(f"Loading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Get config
    if "config" in checkpoint and config_path is None:
        config = OmegaConf.create(checkpoint["config"])
        print("  Using config from checkpoint")
    elif config_path is not None:
        config = OmegaConf.load(config_path)
        print(f"  Using config from: {config_path}")
    else:
        config = OmegaConf.load(CONFIG["config_path"])
        print(f"  Using default config")
    
    # Create model
    model = CLIP4CAD_GFA(config)
    
    # Load weights
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)
    
    model = model.to(device)
    model.eval()
    
    # Print checkpoint info
    epoch = checkpoint.get("epoch", "unknown")
    stage = checkpoint.get("stage", "unknown")
    print(f"  Epoch: {epoch}, Stage: {stage}")
    print(f"  Parameters: {model.count_parameters():,}")
    
    return model, config

# Load model
model, config = load_model(CONFIG["checkpoint_path"], CONFIG.get("config_path"))
print("\nModel loaded successfully!")

In [ ]:
# Cell 3: Generate Embeddings for Validation Set (Queries)

@torch.no_grad()
def generate_embeddings(dataloader, model, desc="Generating embeddings"):
    """Generate embeddings for all samples in the dataloader."""
    text_embeddings = []
    brep_embeddings = []
    pc_embeddings = []
    sample_ids = []
    
    model.eval()
    
    for batch in tqdm(dataloader, desc=desc):
        # Get sample IDs
        batch_ids = batch["sample_id"]
        sample_ids.extend(batch_ids)
        
        # Convert FP16 tensors to FP32 before forward pass
        for key in batch:
            if isinstance(batch[key], torch.Tensor) and batch[key].dtype == torch.float16:
                batch[key] = batch[key].float()
        
        # Forward pass
        outputs = model(batch)
        
        # Extract and normalize embeddings
        z_text = F.normalize(outputs["z_text"].float(), p=2, dim=-1)
        text_embeddings.append(z_text.cpu())
        
        if "z_brep" in outputs:
            z_brep = F.normalize(outputs["z_brep"].float(), p=2, dim=-1)
            brep_embeddings.append(z_brep.cpu())
        
        if "z_pc" in outputs:
            z_pc = F.normalize(outputs["z_pc"].float(), p=2, dim=-1)
            pc_embeddings.append(z_pc.cpu())
    
    result = {
        "text_embeddings": torch.cat(text_embeddings, dim=0),
        "sample_ids": sample_ids,
    }
    
    if brep_embeddings:
        result["brep_embeddings"] = torch.cat(brep_embeddings, dim=0)
    
    if pc_embeddings:
        result["pc_embeddings"] = torch.cat(pc_embeddings, dim=0)
    
    return result


# Create validation dataset and dataloader
print("\n=== Loading Validation Set (Queries) ===")

# Use SSD text file explicitly
val_text_file = Path(CONFIG["text_splits_dir"]) / "val_text_embeddings.h5"
print(f"Using text file: {val_text_file}")

val_dataset = GFAMappedDataset(
    data_root=CONFIG["data_root"],
    split="val",
    pc_file=CONFIG.get("pc_file"),
    text_file=str(val_text_file),  # Explicit SSD path
    mapping_dir=CONFIG["mapping_dir"],
    num_rotations=1,
    load_to_memory=CONFIG.get("load_val_to_memory", True),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=gfa_collate_fn,
)

print(f"Validation samples: {len(val_dataset)}")

# Generate query embeddings
query_data = generate_embeddings(val_loader, model, desc="Generating query embeddings")

query_text_emb = query_data["text_embeddings"]
query_brep_emb = query_data.get("brep_embeddings")
query_pc_emb = query_data.get("pc_embeddings")
query_ids = query_data["sample_ids"]

print(f"\n✓ Query embeddings generated: {len(query_ids)} samples")
print(f"  Text: {query_text_emb.shape}")
if query_brep_emb is not None:
    print(f"  B-Rep: {query_brep_emb.shape}")
if query_pc_emb is not None:
    print(f"  PC: {query_pc_emb.shape}")

In [ ]:
# Cell 4: Generate Embeddings for Training Set (Gallery)

print("\n=== Loading Training Set (Gallery) ===")

# Use SSD text file explicitly
train_text_file = Path(CONFIG["text_splits_dir"]) / "train_text_embeddings.h5"
print(f"Using text file: {train_text_file}")

train_dataset = GFAMappedDataset(
    data_root=CONFIG["data_root"],
    split="train",
    pc_file=CONFIG.get("pc_file"),
    text_file=str(train_text_file),  # Explicit SSD path
    mapping_dir=CONFIG["mapping_dir"],
    num_rotations=1,
    load_to_memory=CONFIG.get("load_train_to_memory", False),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=gfa_collate_fn,
)

print(f"Training samples: {len(train_dataset)}")

# Generate gallery embeddings
gallery_data = generate_embeddings(train_loader, model, desc="Generating gallery embeddings")

gallery_text_emb = gallery_data["text_embeddings"]
gallery_brep_emb = gallery_data.get("brep_embeddings")
gallery_pc_emb = gallery_data.get("pc_embeddings")
gallery_ids = gallery_data["sample_ids"]

print(f"\n✓ Gallery embeddings generated: {len(gallery_ids)} samples")
print(f"  Text: {gallery_text_emb.shape}")
if gallery_brep_emb is not None:
    print(f"  B-Rep: {gallery_brep_emb.shape}")
if gallery_pc_emb is not None:
    print(f"  PC: {gallery_pc_emb.shape}")

# Close datasets
val_dataset.close()
train_dataset.close()

In [ ]:
# Cell 5: Compute Rankings (Memory-Efficient)

def compute_top_k_rankings(query_embeddings, gallery_embeddings, k=50, batch_size=64):
    """
    Compute top-k rankings for all queries in a memory-efficient way.
    
    Args:
        query_embeddings: (N_q, D) tensor
        gallery_embeddings: (N_g, D) tensor
        k: Number of top results to keep
        batch_size: Batch size for similarity computation
    
    Returns:
        rankings: (N_q, k) tensor of gallery indices
    """
    num_queries = query_embeddings.shape[0]
    rankings = torch.zeros(num_queries, k, dtype=torch.long)
    
    # Move gallery to GPU once
    gallery_gpu = gallery_embeddings.to(device)
    
    for i in tqdm(range(0, num_queries, batch_size), desc=f"Computing top-{k} rankings"):
        end_idx = min(i + batch_size, num_queries)
        query_batch = query_embeddings[i:end_idx].to(device)
        
        # Compute similarities (cosine similarity since embeddings are normalized)
        similarities = torch.mm(query_batch, gallery_gpu.T)
        
        # Get top k
        _, top_indices = torch.topk(similarities, k=k, dim=1)
        rankings[i:end_idx] = top_indices.cpu()
    
    return rankings


# Compute rankings for each task
k = max(CONFIG["k_values"])
batch_size = 128  # Larger batch for ranking computation

print("\n=== Computing Rankings ===")

# Task 1: Text → B-Rep
print("\nTask 1: Text → B-Rep")
if gallery_brep_emb is not None:
    text_to_brep_rankings = compute_top_k_rankings(
        query_text_emb, gallery_brep_emb, k, batch_size
    )
else:
    print("  Skipped (no B-Rep embeddings)")
    text_to_brep_rankings = None

# Task 2: Text → Point Cloud
print("\nTask 2: Text → Point Cloud")
if gallery_pc_emb is not None:
    text_to_pc_rankings = compute_top_k_rankings(
        query_text_emb, gallery_pc_emb, k, batch_size
    )
else:
    print("  Skipped (no PC embeddings)")
    text_to_pc_rankings = None

# Task 3: Point Cloud → B-Rep
print("\nTask 3: Point Cloud → B-Rep")
if query_pc_emb is not None and gallery_brep_emb is not None:
    pc_to_brep_rankings = compute_top_k_rankings(
        query_pc_emb, gallery_brep_emb, k, batch_size
    )
else:
    print("  Skipped (missing embeddings)")
    pc_to_brep_rankings = None

print("\n✓ Rankings computed")

In [ ]:
# Cell 6: Compute Evaluation Metrics

def compute_mAP_at_k(rankings, query_ids, gallery_ids, k):
    """
    Compute mAP@k using sample_id matching.
    A sample is relevant if it has the same sample_id.
    
    Args:
        rankings: (N_q, K) tensor of gallery indices
        query_ids: List of query sample IDs
        gallery_ids: List of gallery sample IDs
        k: Number of top results to consider
    """
    num_queries = rankings.shape[0]
    ap_sum = 0.0
    
    for i in range(num_queries):
        query_id = query_ids[i]
        top_k_indices = rankings[i, :k]
        
        # Calculate AP for this query
        precision_sum = 0.0
        num_relevant = 0
        
        for j, pred_idx in enumerate(top_k_indices):
            pred_id = gallery_ids[pred_idx]
            
            # Check if same sample (relevant)
            if str(pred_id) == str(query_id):
                num_relevant += 1
                precision_at_j = num_relevant / (j + 1)
                precision_sum += precision_at_j
        
        # Average precision for this query
        if num_relevant > 0:
            ap = precision_sum / num_relevant
            ap_sum += ap
    
    return ap_sum / num_queries


def compute_recall_at_k(rankings, query_ids, gallery_ids, k):
    """
    Compute Recall@k using sample_id matching.
    """
    num_queries = rankings.shape[0]
    correct = 0
    
    for i in range(num_queries):
        query_id = query_ids[i]
        top_k_indices = rankings[i, :k]
        
        # Check if any of top-k matches the query
        for pred_idx in top_k_indices:
            pred_id = gallery_ids[pred_idx]
            
            if str(pred_id) == str(query_id):
                correct += 1
                break  # Count once per query
    
    return correct / num_queries


def evaluate_task(task_name, rankings, query_ids, gallery_ids, k_values):
    """Evaluate a retrieval task and print results."""
    if rankings is None:
        print(f"\n{task_name}: Skipped (missing embeddings)")
        return None
    
    print(f"\n{'-'*60}")
    print(f"{task_name}")
    print(f"{'-'*60}")
    
    results = {}
    
    for k in k_values:
        map_k = compute_mAP_at_k(rankings, query_ids, gallery_ids, k)
        recall_k = compute_recall_at_k(rankings, query_ids, gallery_ids, k)
        
        results[f"mAP@{k}"] = map_k
        results[f"R@{k}"] = recall_k
        
        print(f"mAP@{k:2d}: {map_k:.4f}  |  R@{k:2d}: {recall_k:.4f}")
    
    return results


# Evaluate all tasks
k_values = CONFIG["k_values"]

print("\n" + "="*60)
print("EVALUATION RESULTS - CLIP4CAD-GFA")
print("="*60)

print(f"\nQuery samples: {len(query_ids)}")
print(f"Gallery samples: {len(gallery_ids)}")

all_results = {}

# Text → B-Rep
results = evaluate_task(
    "Text → B-Rep",
    text_to_brep_rankings,
    query_ids,
    gallery_ids,
    k_values,
)
if results:
    all_results["text_to_brep"] = results

# Text → Point Cloud
results = evaluate_task(
    "Text → Point Cloud",
    text_to_pc_rankings,
    query_ids,
    gallery_ids,
    k_values,
)
if results:
    all_results["text_to_pc"] = results

# Point Cloud → B-Rep
results = evaluate_task(
    "Point Cloud → B-Rep",
    pc_to_brep_rankings,
    query_ids,
    gallery_ids,
    k_values,
)
if results:
    all_results["pc_to_brep"] = results

print("\n" + "="*60)

In [ ]:
# Cell 7: Save Summary Results

# Create output directory
output_dir = Path(CONFIG["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

# Build results summary
summary = {
    "model": CONFIG["checkpoint_path"],
    "config": CONFIG.get("config_path", "from_checkpoint"),
    "num_queries": len(query_ids),
    "num_gallery": len(gallery_ids),
    "metrics": all_results,
}

# Save to JSON
results_path = output_dir / "evaluation_results.json"
with open(results_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Results saved to: {results_path}")

# Print formatted table for paper
print("\n" + "="*80)
print("RESULTS TABLE (for paper)")
print("="*80)
print(f"\n{'Task':<20} {'mAP@1':<10} {'mAP@5':<10} {'mAP@10':<10} {'R@1':<10} {'R@5':<10} {'R@10':<10}")
print("-"*80)

for task_name, metrics in all_results.items():
    display_name = task_name.replace("_", " → ").replace("text", "Text").replace("brep", "B-Rep").replace("pc", "PC")
    print(f"{display_name:<20} "
          f"{metrics.get('mAP@1', 0):<10.4f} "
          f"{metrics.get('mAP@5', 0):<10.4f} "
          f"{metrics.get('mAP@10', 0):<10.4f} "
          f"{metrics.get('R@1', 0):<10.4f} "
          f"{metrics.get('R@5', 0):<10.4f} "
          f"{metrics.get('R@10', 0):<10.4f}")

print("\n" + "="*80)

In [ ]:
# Cell 8: Optional - Save Embeddings for Later Use

SAVE_EMBEDDINGS = False  # Set to True to save embeddings

if SAVE_EMBEDDINGS:
    print("Saving embeddings...")
    
    # Save query embeddings
    torch.save({
        "text_embeddings": query_text_emb,
        "brep_embeddings": query_brep_emb,
        "pc_embeddings": query_pc_emb,
        "sample_ids": query_ids,
    }, output_dir / "val_embeddings.pt")
    
    # Save gallery embeddings
    torch.save({
        "text_embeddings": gallery_text_emb,
        "brep_embeddings": gallery_brep_emb,
        "pc_embeddings": gallery_pc_emb,
        "sample_ids": gallery_ids,
    }, output_dir / "train_embeddings.pt")
    
    # Save rankings
    torch.save({
        "text_to_brep": text_to_brep_rankings,
        "text_to_pc": text_to_pc_rankings,
        "pc_to_brep": pc_to_brep_rankings,
    }, output_dir / "rankings.pt")
    
    print(f"✓ Embeddings saved to: {output_dir}")
else:
    print("Embeddings not saved. Set SAVE_EMBEDDINGS = True to save.")

In [ ]:
# Cell 9: Optional - Visualization of Top Retrievals

def visualize_retrieval(query_idx, rankings, query_ids, gallery_ids, task_name, k=5):
    """
    Print retrieval results for a single query.
    """
    query_id = query_ids[query_idx]
    top_k = rankings[query_idx, :k]
    
    print(f"\n{task_name} - Query #{query_idx}")
    print(f"  Query ID: {query_id}")
    print(f"  Top-{k} Results:")
    
    for rank, pred_idx in enumerate(top_k, 1):
        pred_id = gallery_ids[pred_idx]
        match = "✓" if str(pred_id) == str(query_id) else "✗"
        print(f"    {rank}. {pred_id} {match}")


# Show example retrievals
print("\n" + "="*60)
print("EXAMPLE RETRIEVALS")
print("="*60)

# Random query indices
np.random.seed(42)
example_indices = np.random.choice(len(query_ids), min(3, len(query_ids)), replace=False)

for idx in example_indices:
    if text_to_brep_rankings is not None:
        visualize_retrieval(idx, text_to_brep_rankings, query_ids, gallery_ids, "Text → B-Rep")
    if text_to_pc_rankings is not None:
        visualize_retrieval(idx, text_to_pc_rankings, query_ids, gallery_ids, "Text → PC")

## Summary

This notebook evaluated the CLIP4CAD-GFA model on cross-modal retrieval tasks:

1. **Text → B-Rep**: Find B-Rep models matching a text description
2. **Text → Point Cloud**: Find point clouds matching a text description  
3. **Point Cloud → B-Rep**: Find B-Rep models matching a point cloud

The metrics (mAP@k, Recall@k) measure how well the model aligns different modalities in the shared embedding space.

### Notes

- Relevance is determined by **sample_id matching** (same CAD model = relevant)
- All embeddings are L2-normalized, so cosine similarity equals dot product
- Results are saved to `outputs/evaluation/evaluation_results.json`